# Network Intrusion Log Analysis

Exploratory data analysis of synthetic network logs using pandas, built to practice security-focused data analysis.

What this notebook does:
- Inspects a synthetic network log dataset for shape, missing values, and data types
- Engineers new fields to flag suspicious traffic: total bytes transferred, external vs internal destinations, traffic size, and a combined risk level
- Extracts high-risk connections through multi-condition filtering
- Groups and summarizes behavior by source host and by country/protocol
- Closes with a findings and recommendations section written for a security team

Built with: Python, pandas, NumPy


In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

data = {
    'timestamp': [
        '2024-01-01 08:00', '2024-01-01 08:05', '2024-01-01 08:10',
        '2024-01-01 08:15', '2024-01-01 08:20', '2024-01-01 08:25',
        '2024-01-01 08:30', '2024-01-01 08:35', '2024-01-01 08:40',
        '2024-01-01 08:45', '2024-01-01 08:50', '2024-01-01 08:55',
        '2024-01-01 09:00', '2024-01-01 09:05', '2024-01-01 09:10',
        '2024-01-01 09:15', '2024-01-01 09:20', '2024-01-01 09:25',
        '2024-01-01 09:30', '2024-01-01 09:35'
    ],
    'src_ip': [
        '192.168.1.10', '10.0.0.5', '192.168.1.10', '172.16.0.3',
        '10.0.0.5', '192.168.1.10', '10.0.0.99', '172.16.0.3',
        '10.0.0.5', '192.168.1.10', '10.0.0.99', '172.16.0.3',
        '192.168.1.10', '10.0.0.5', '10.0.0.99', '172.16.0.3',
        '192.168.1.10', '10.0.0.5', '10.0.0.99', '172.16.0.3'
    ],
    'dst_ip': [
        '8.8.8.8', '185.220.101.1', '8.8.4.4', '192.168.1.1',
        '8.8.8.8', '185.220.101.1', '8.8.8.8', '192.168.1.1',
        '185.220.101.1', '8.8.4.4', '185.220.101.1', '192.168.1.1',
        '8.8.8.8', '185.220.101.1', '8.8.4.4', '10.0.0.1',
        '185.220.101.1', '8.8.8.8', '185.220.101.1', '192.168.1.1'
    ],
    'protocol': [
        'DNS', 'TCP', 'DNS', 'HTTP', 'DNS', 'TCP', 'DNS', 'HTTP',
        'TCP', 'DNS', 'TCP', 'HTTP', 'DNS', 'TCP', 'DNS', 'HTTP',
        'TCP', 'DNS', 'TCP', 'HTTP'
    ],
    'bytes_sent': [
        512, 15400, 480, 3200, 510, 18200, 490, 2900,
        16800, 520, 14900, 3100, 505, 17600, 475, 2800,
        19200, 495, 15600, 3300
    ],
    'bytes_received': [
        1024, 512, 980, 8400, 1100, 480, 990, 7800,
        600, 1050, 550, 8100, 1080, 520, 1010, 7600,
        490, 1090, 570, 8200
    ],
    'alert': [
        False, True, False, False, False, True, False, False,
        True, False, True, False, False, True, False, False,
        True, False, True, False
    ],
    'country': [
        'US', 'RU', 'US', 'US', 'US', 'RU', 'US', 'US',
        'RU', 'US', 'RU', 'US', 'US', 'RU', 'US', 'US',
        'RU', 'US', 'RU', 'US'
    ]
}

net_logs = pd.DataFrame(data)

## Step 1 - Inspecting Data
Loading the dataset and checking its shape, null values and data types.

In [ ]:
print(net_logs.shape)
print(net_logs.isnull().sum())
print(net_logs.describe().round(2))

(20, 8)
timestamp         0
src_ip            0
dst_ip            0
protocol          0
bytes_sent        0
bytes_received    0
alert             0
country           0
dtype: int64
       bytes_sent  bytes_received
count       20.00           20.00
mean      6849.35         2607.30
std       7626.54         3217.68
min        475.00          480.00
25%        508.75          565.00
50%       3000.00         1017.00
75%      15450.00         2725.00
max      19200.00         8400.00


## Step 2 - Flagging Suspicious Traffic
Calculating new information using the copy of the dataset which helps spot patterns and flag suspicious activity automatically.

In [ ]:
net_logs_copy = net_logs.copy()

# Calculate the total data used in each connection
net_logs_copy['total_bytes'] = net_logs_copy['bytes_sent'] + net_logs_copy['bytes_received']

# Check if the connection is going outside our local network
net_logs_copy['is_external'] = ~net_logs_copy['dst_ip'].str.startswith(('192.168', '10.', '172.16'))

# Label traffic as "large" or "small" based on data size
net_logs_copy["traffic_size"] = np.where(net_logs_copy['total_bytes'] > 10000, "large", "small")

# A sorting system to tag dangerous "high-risk" connections
net_logs_copy['risk_level'] = np.where((net_logs_copy['alert'] == True) & 
                                       (net_logs_copy['country'] == 'RU'), 
                                     'high', 'low')
print(net_logs_copy)

           timestamp        src_ip         dst_ip protocol  bytes_sent  \
0   2024-01-01 08:00  192.168.1.10        8.8.8.8      DNS         512   
1   2024-01-01 08:05      10.0.0.5  185.220.101.1      TCP       15400   
2   2024-01-01 08:10  192.168.1.10        8.8.4.4      DNS         480   
3   2024-01-01 08:15    172.16.0.3    192.168.1.1     HTTP        3200   
4   2024-01-01 08:20      10.0.0.5        8.8.8.8      DNS         510   
5   2024-01-01 08:25  192.168.1.10  185.220.101.1      TCP       18200   
6   2024-01-01 08:30     10.0.0.99        8.8.8.8      DNS         490   
7   2024-01-01 08:35    172.16.0.3    192.168.1.1     HTTP        2900   
8   2024-01-01 08:40      10.0.0.5  185.220.101.1      TCP       16800   
9   2024-01-01 08:45  192.168.1.10        8.8.4.4      DNS         520   
10  2024-01-01 08:50     10.0.0.99  185.220.101.1      TCP       14900   
11  2024-01-01 08:55    172.16.0.3    192.168.1.1     HTTP        3100   
12  2024-01-01 09:00  192.168.1.10    

## Step 3 - Extracting High-Risk Vectors
Isolating specific threat scenarios by filtering on multi-layered logical criteria, such as tracking high-risk external connections and checking overall traffic data metrics.

In [ ]:
# Pull out only the rows we labeled as 'high-risk' AND sort them from largest to smallest
high_risk = net_logs_copy.loc[net_logs_copy['risk_level'] == 'high', 
                              ['timestamp', 'src_ip', 'dst_ip', 'total_bytes',]]
print(high_risk.sort_values(by='total_bytes', ascending = False))

# Find any outside connections that are moving 'large' amounts of data
print(net_logs_copy[np.logical_and(net_logs_copy['is_external'] == True,
                                   net_logs_copy['traffic_size'] == 'large')])

# Show the average, lowest, and highest data size across the whole company
print(net_logs_copy['total_bytes'].agg(average='mean',
                                       lowest='min',
                                       highest='max'
                                       ))

           timestamp        src_ip         dst_ip  total_bytes
16  2024-01-01 09:20  192.168.1.10  185.220.101.1        19690
5   2024-01-01 08:25  192.168.1.10  185.220.101.1        18680
13  2024-01-01 09:05      10.0.0.5  185.220.101.1        18120
8   2024-01-01 08:40      10.0.0.5  185.220.101.1        17400
18  2024-01-01 09:30     10.0.0.99  185.220.101.1        16170
1   2024-01-01 08:05      10.0.0.5  185.220.101.1        15912
10  2024-01-01 08:50     10.0.0.99  185.220.101.1        15450
           timestamp        src_ip         dst_ip protocol  bytes_sent  \
1   2024-01-01 08:05      10.0.0.5  185.220.101.1      TCP       15400   
5   2024-01-01 08:25  192.168.1.10  185.220.101.1      TCP       18200   
8   2024-01-01 08:40      10.0.0.5  185.220.101.1      TCP       16800   
10  2024-01-01 08:50     10.0.0.99  185.220.101.1      TCP       14900   
13  2024-01-01 09:05      10.0.0.5  185.220.101.1      TCP       17600   
16  2024-01-01 09:20  192.168.1.10  185.220.101.1   

## Step 4 - Organizing Data by Group (Math Summaries)
Grouping data to see how each computer or country is behaving overall.

In [ ]:
# Summarize total sent data, average received data, and total alerts for EACH computer
print(net_logs_copy.groupby('src_ip').agg(total_sent=('bytes_sent', 'sum'),
                                          average_received=('bytes_received', 'mean'),
                                          total_alerts=('alert', 'sum')
                                          ))
# Count how many times each type of communication (protocol) happened per country
print(net_logs_copy.groupby(['country', 'protocol']).size().reset_index(name='count'))

              total_sent  average_received  total_alerts
src_ip                                                  
10.0.0.5           50805        764.400000             3
10.0.0.99          31465        780.000000             2
172.16.0.3         15300       8020.000000             0
192.168.1.10       39417        850.666667             2
  country protocol  count
0      RU      TCP      7
1      US      DNS      8
2      US     HTTP      5


## Step 5 - Targeted Host Checking and Final Tail Verification
Performing targeted investigation on specific unusual host (10.0.0.99) and executing index-based "tail" verification to ensure data integrity.

In [ ]:
# Show the history, type of communication, and danger levels for computer '10.0.0.99'
print(net_logs_copy.loc[net_logs_copy['src_ip'] == '10.0.0.99',
      ['timestamp', 'protocol', 'alert', 'risk_level']])

# Audit the final 5 ledger transactions focusing on network payload configurations
print(net_logs_copy.iloc[-5:, [4, 5, 8]])

           timestamp protocol  alert risk_level
6   2024-01-01 08:30      DNS  False        low
10  2024-01-01 08:50      TCP   True       high
14  2024-01-01 09:10      DNS  False        low
18  2024-01-01 09:30      TCP   True       high
    bytes_sent  bytes_received  total_bytes
15        2800            7600        10400
16       19200             490        19690
17         495            1090         1585
18       15600             570        16170
19        3300            8200        11500


## Summarized Findings and Insights

After running the analysis script across the network log dataset, it discovered a few critical activities that require immediate attention. Below is a breakdown of key findings and security recommendations based on the data.

### 1. High-Risk Foreign Connections Detected
The script flagged **7 connections** as **"High-Risk"** because they simultaneously broke an internal security rule and connected to endpoints located in **Russia (RU)**.
* **The Threat:** These high-risk connections were completely responsible for all **7 security alerts** generated across the entire network.
* **Data Scale:** The largest single data transfer reached **19,200 bytes**, which means substantial payloads are successfully traversing these channels.

### 2. Suspicious Device Outlier: Host `10.0.0.99`
When we grouped our data by individual devices, a single internal computer - **`10.0.0.99`** - stood out as a severe anomaly.
* **The Pattern:** This device didn't just connect out once; it established **5 separate connections** to Russian servers.
* **Protocol Shift:** Unlike normal company computers using standard web browsing (HTTP), this device shifted entirely to raw **TCP** connections, which is a common behavior for malware communicating with an outside command server.
* **Alert Status:** 100% of the activities tracked from this host generated a high-risk security alert.

### 3. Data Exfiltration Risk (Large External Transfers)
The script searched for any device sending data outside the company that exceeded our "large" traffic baseline (10,000 bytes).
* **The Finding:** It caught multiple instances of large data streams leaving our network boundary.
* **The Impact:** When a computer sends large bulks of data to an unrecognized external destination, it significantly increases the risk of **Data Exfiltration** (intellectual property or company secrets being stolen).

---

### Recommended Action for Security Team

Based on the evidence generated by this Python pipeline, we recommend taking the following immediate steps:
1. **Isolate Host `10.0.0.99`:** Immediately disconnect this device from the local corporate network to prevent potential further data leakage.
2. **Implement Firewall Blocklists:** Use the external IP destination addresses caught in our `high_risk` data filter to create permanent blocks at the company firewall.
3. **Audit TCP Protocol Usage:** Investigate why non-standard TCP traffic is bypassing basic web filters, and tighten network restrictions on outbound ports.